In [ ]:
import nltk
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [ ]:
!pip install langdetect

In [ ]:
import pandas as pd
import numpy as np
import re
import seaborn as sns
from nltk.corpus import stopwords
from nltk.tokenize import sent_tokenize
from nltk.stem import PorterStemmer
from langdetect import detect, LangDetectException

In [ ]:
df = pd.read_csv('data_label_manual.csv')
print("="*35)

print(df.head().to_markdown(index=False, numalign="left", stralign="left"))
print("="*35)

print(df.info())
print("="*35)

display(df)

| Ulasan_Bersih;Labels;                                                                         |
|:----------------------------------------------------------------------------------------------|
| halo selamat malam admin anime one piece episode kenapa mohon ditanggapi terima kasih;Netral; |
| rating seberapa login susah minta ampun mahal bayar kocak pantes sepi;Negatif;                |
| payah dibuka percuma donload;Negatif;                                                         |
| good;Positif;                                                                                 |
| kualitas flim bagus sekali;Positif;                                                           |
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 909 entries, 0 to 908
Data columns (total 1 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   Ulasan_Bersih;Labels;  909 non-null    object
dtypes: object(1)
memory usage: 7.2+ KB
None


,Ulasan_Bersih;Labels;
0,halo selamat malam admin anime one piece episo...
1,rating seberapa login susah minta ampun mahal ...
2,payah dibuka percuma donload;Negatif;
3,good;Positif;
4,kualitas flim bagus sekali;Positif;
...,...
904,minta refund uang sudah langgan masuk eror ter...
905,sudah berlanggan baru hari tiba tiba dapat con...
906,bingung pendaftaran disuruh bayar dulu cok pad...
907,buruk;;


In [ ]:
# Define the logic for sentiment
def extract_sentiment_label(combined_text):
    if pd.isna(combined_text):
        return None
    parts = str(combined_text).split(';')
    if len(parts) > 1:
        potential_label = parts[1].strip()
        # Memastikan label sentimen adalah salah satu yang diharapkan
        if potential_label in ['Netral', 'Negatif', 'Positif']:
            return potential_label
        # Beberapa baris mungkin memiliki label sentimen pada indeks 2 atau lebih jika bagian sebelumnya kosong
        elif len(parts) > 2 and parts[2].strip() in ['Netral', 'Negatif', 'Positif']:
            return parts[2].strip()
        # Fallback untuk kasus di mana label sentimen tidak langsung ada di indeks 1 atau 2
        # namun masih ada dalam bagian lainnya
        elif 'Negatif' in parts:
            return 'Negatif'
        elif 'Positif' in parts:
            return 'Positif'
        elif 'Netral' in parts:
            return 'Netral'
    return None  # Jika tidak ditemukan label sentimen yang jelas

# Terapkan fungsi ekstraksi untuk membuat kolom 'sentiment'
# df.columns[0] merujuk pada kolom pertama (dan satu-satunya) 'Ulasan_Bersih;Labels;'
df['sentiment'] = df[df.columns[0]].apply(extract_sentiment_label)

# Hapus baris di mana label sentimen tidak dapat diekstraksi dengan jelas
# karena ini akan menjadi masalah untuk pengambilan sampel
df.dropna(subset=['sentiment'], inplace=True)

# Pisahkan ulasan berdasarkan sentimen yang terdeteksi
negative_reviews = df[df['sentiment'] == 'Negatif']
neutral_reviews = df[df['sentiment'] == 'Netral']
positive_reviews = df[df['sentiment'] == 'Positif']

# Tangani kasus di mana jumlah sampel tidak mencukupi untuk kategori tertentu
# Gunakan min() untuk memastikan kita tidak mencoba mengambil sampel lebih banyak dari jumlah yang tersedia
sampled_positive = positive_reviews.sample(n=min(100, len(positive_reviews)), random_state=42)
sampled_negative = negative_reviews.sample(n=min(100, len(negative_reviews)), random_state=42)
sampled_neutral = neutral_reviews.sample(n=min(50, len(neutral_reviews)), random_state=42)

# Gabungkan sampel dari ketiga kategori menjadi satu DataFrame
final_df_500 = pd.concat([sampled_positive, sampled_negative, sampled_neutral])
# Acak urutan baris dan reset indeks
final_df_500 = final_df_500.sample(frac=1, random_state=42).reset_index(drop=True)
# Simpan hasil pengambilan sampel ke dalam file CSV
final_df_500.to_csv('data_label_training.csv', index=False)

# Cetak pembatas untuk pemisahan output
print("="*35)

# Tampilkan jumlah sampel per kategori sentimen
print(final_df_500['sentiment'].value_counts())
print("="*35)

# Tampilkan informasi tentang DataFrame akhir
print(final_df_500.info())


sentiment
Negatif    100
Positif    100
Netral      50
Name: count, dtype: int64
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250 entries, 0 to 249
Data columns (total 2 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   Ulasan_Bersih;Labels;  250 non-null    object
 1   sentiment              250 non-null    object
dtypes: object(2)
memory usage: 4.0+ KB
None


In [ ]:
from nltk.tokenize import sent_tokenize

data = final_df_500['Ulasan_Bersih;Labels;'].apply(sent_tokenize)

total_documents = len(data)
print(f"\nTotal dokumen yang ditokenisasi: {total_documents}")
print(data.head())


Total dokumen yang ditokenisasi: 250
0    [kualitas gambar atur langsung kaya dulu penur...
1                       [sangat bagus sekali;Positif;]
2                                     [bagus;Positif;]
3                                   [berguna;Positif;]
4    [mending nonton steel bal run website baja dar...
Name: Ulasan_Bersih;Labels;, dtype: object


In [20]:
stemmer = PorterStemmer()
stop_words = stopwords.words('english') # Note: Consider using Indonesian stopwords if reviews are in Indonesian.

def preprocess(text):
  if not isinstance(text, str) or pd.isna(text):
    return ""
  text = re.sub(r'<[^>]+>|https?://\S+|[^a-zA-Z0-9\s]', ' ', text.lower())
  tokens = text.split()
  filtered = [w for w in tokens if w not in stop_words]
  stemmed = [stemmer.stem(word) for word in filtered]
  return ' '.join(stemmed)

# Extract the actual review text from 'Ulasan_Bersih;Labels;' and create a new 'review_text' column
final_df_500['review_text'] = final_df_500['Ulasan_Bersih;Labels;'].apply(lambda x: str(x).split(';')[0].strip())

# Apply the preprocess function to the new 'review_text' column
final_df_500['processed_review_text'] = final_df_500['review_text'].apply(preprocess)

# Display the head with the new 'review_text' and 'processed_review_text' columns
print(final_df_500[['review_text', 'sentiment', 'processed_review_text']].head())

# The original loop for 'rating' is removed as there is no 'rating' column in final_df_500.
# Instead, let's show some processed samples for each sentiment category.
print("\n--- Sampel Ulasan Positif Terproses ---")
print(final_df_500[final_df_500['sentiment'] == 'Positif'][['review_text', 'processed_review_text']].head(2).to_markdown(index=False, numalign="left", stralign="left"))

print("\n--- Sampel Ulasan Negatif Terproses ---")
print(final_df_500[final_df_500['sentiment'] == 'Negatif'][['review_text', 'processed_review_text']].head(2).to_markdown(index=False, numalign="left", stralign="left"))

print("\n--- Sampel Ulasan Netral Terproses ---")
print(final_df_500[final_df_500['sentiment'] == 'Netral'][['review_text', 'processed_review_text']].head(2).to_markdown(index=False, numalign="left", stralign="left"))

                                         review_text sentiment  \
0  kualitas gambar atur langsung kaya dulu penuru...   Negatif   
1                                sangat bagus sekali   Positif   
2                                              bagus   Positif   
3                                            berguna   Positif   
4  mending nonton steel bal run website baja dari...   Negatif   

                               processed_review_text  
0  kualita gambar atur langsung kaya dulu penurun...  
1                                 sangat bagu sekali  
2                                               bagu  
3                                            berguna  
4     mend nonton steel bal run websit baja daripada  

--- Sampel Ulasan Positif Terproses ---
| review_text         | processed_review_text   |
|:--------------------|:------------------------|
| sangat bagus sekali | sangat bagu sekali      |
| bagus               | bagu                    |

--- Sampel Ulasan Negatif Terpr

In [21]:
from langdetect import detect, LangDetectException

# --- BAGIAN 1: FUNGSI FILTER BAHASA --- dan EKSTRAKSI SENTIMEN
def is_english(text):
    try:
        # Langdetect bisa gagal pada string pendek atau karakter non-alfabet, pastikan string valid terlebih dahulu
        if not isinstance(text, str) or not text.strip():
            return False
        return detect(text) == 'en'
    except LangDetectException:
        return False

def extract_sentiment_label_from_combined(combined_text):
    if pd.isna(combined_text):
        return None
    parts = str(combined_text).split(';')
    # Iterasi melalui bagian untuk menemukan label sentimen
    for part in parts:
        stripped_part = part.strip()
        if stripped_part in ['Netral', 'Negatif', 'Positif']:
            return stripped_part
    return None

# --- BAGIAN 2: LOAD & FILTER DATA ---
df = pd.read_csv('data_label_manual.csv')
print("Jumlah data awal:", len(df))

# Ubah nama kolom tunggal untuk akses dan kejelasan yang lebih mudah
original_col_name = df.columns[0]
df.rename(columns={original_col_name: 'ulasan_labels_combined'}, inplace=True)

# Ekstrak teks ulasan dan label sentimen
df['review_text'] = df['ulasan_labels_combined'].apply(lambda x: str(x).split(';')[0].strip())
df['sentiment'] = df['ulasan_labels_combined'].apply(extract_sentiment_label_from_combined)

# Hapus baris di mana sentimen tidak dapat diekstraksi dengan jelas (misalnya, baris yang tidak terformat)
df.dropna(subset=['sentiment'], inplace=True)

print("Sedang memfilter bahasa Inggris (mohon tunggu)...")
# Terapkan is_english ke review_text yang diekstrak
df = df[df['review_text'].apply(is_english)]
print("Jumlah data setelah filter bahasa Inggris:", len(df))

# --- BAGIAN 3: SAMPLING BERDASARKAN SENTIMEN ---
# Pisahkan berdasarkan sentimen
negative_reviews = df[df['sentiment'] == 'Negatif']
neutral_reviews = df[df['sentiment'] == 'Netral']
positive_reviews = df[df['sentiment'] == 'Positif']

try:
    # Gunakan min() untuk memastikan kita tidak mencoba mengambil sampel lebih banyak dari yang tersedia
    sampled_positive = positive_reviews.sample(n=min(200, len(positive_reviews)), random_state=42)
    sampled_negative = negative_reviews.sample(n=min(200, len(negative_reviews)), random_state=42)
    sampled_neutral = neutral_reviews.sample(n=min(100, len(neutral_reviews)), random_state=42)
except ValueError:
    print("Peringatan: Data bahasa Inggris tidak cukup untuk memenuhi target kuota.")
    sampled_positive = positive_reviews.head(min(200, len(positive_reviews)))
    sampled_negative = negative_reviews.head(min(200, len(negative_reviews)))
    sampled_neutral = neutral_reviews.head(min(100, len(neutral_reviews)))

# Gabungkan & Acak
final_df = pd.concat([sampled_positive, sampled_negative, sampled_neutral])
final_df = final_df.sample(frac=1, random_state=42).reset_index(drop=True)
final_df.to_csv('sampled_threads_reviews.csv', index=False)

# --- BAGIAN 4: TOKENISASI ---
print("\n--- Proses Tokenisasi ---")
# Terapkan sent_tokenize ke review_text yang diekstrak di final_df
data_tokenized = final_df['review_text'].apply(sent_tokenize)

print(f"Total dokumen final: {len(data_tokenized)}")
print(final_df['sentiment'].value_counts())

# Tampilkan contoh hasil
print("\nContoh data (Bahasa Inggris):")
print(final_df[['review_text', 'sentiment']].head())

Jumlah data awal: 909
Sedang memfilter bahasa Inggris (mohon tunggu)...
Jumlah data setelah filter bahasa Inggris: 4

--- Proses Tokenisasi ---
Total dokumen final: 4
sentiment
Positif    3
Netral     1
Name: count, dtype: int64

Contoh data (Bahasa Inggris):
                        review_text sentiment
0                          the best   Positif
1  setiap nonton loading terus aneh    Netral
2                          the best   Positif
3                   recommended apk   Positif


In [22]:
# Load datasets
try:
    data_manual = pd.read_csv('data_label_manual.csv')
    full_data = pd.read_csv('data_label_training.csv')
    print("Data Manual Loaded. Shape:", data_manual.shape)
    print("Full Data Loaded. Shape:", full_data.shape)
    print("Data Manual Columns:", data_manual.columns)
except Exception as e:
    print("Error loading data:", e)

Data Manual Loaded. Shape: (909, 1)
Full Data Loaded. Shape: (250, 2)
Data Manual Columns: Index(['Ulasan_Bersih;Labels;'], dtype='object')


In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import pandas as pd
import re

try:
    from langdetect import detect, LangDetectException
except ImportError:
    pass

# Fungsi bantu untuk mengekstrak teks ulasan dan sentimen dari kolom gabungan
def extract_review_and_sentiment(combined_text):
    if pd.isna(combined_text):
        return None, None
    parts = str(combined_text).split(';')
    review_text = parts[0].strip()
    sentiment = None
    if len(parts) > 1:
        for part in parts[1:]: # Memeriksa semua bagian untuk label sentimen
            stripped_part = part.strip()
            if stripped_part in ['Netral', 'Negatif', 'Positif']:
                sentiment = stripped_part
                break
    return review_text, sentiment

# Fungsi untuk mendeteksi apakah teks berbahasa Inggris
def is_english(text):
    try:
        if not isinstance(text, str) or not text.strip():
            return False
        # Langdetect dapat gagal pada string yang sangat pendek atau non-alfabet
        if len(text.strip()) < 3 or not re.search('[a-zA-Z]', text.strip()):
            return False
        return detect(text) == 'en'
    except LangDetectException:
        return False
    except Exception: # Menangani kesalahan lainnya dari langdetect
        return False

# 1. Memuat Data
try:
    df_train = pd.read_csv('data_label_training.csv')
    df_full_pool = pd.read_csv('data_label_manual.csv')
    print("Data pelatihan dimuat. Bentuk:", df_train.shape)
    print("Pool data penuh dimuat. Bentuk:", df_full_pool.shape)
except FileNotFoundError:
    print("Error: File CSV tidak ditemukan. Pastikan file ada di direktori yang sama.")
    exit()

# Menyiapkan data pelatihan (df_train)
df_train['review_text'] = df_train['Ulasan_Bersih;Labels;'].apply(lambda x: str(x).split(';')[0].strip())
# Menyiapkan pool data penuh (df_full_pool)
# Mengekstrak 'review_text' dari 'Ulasan_Bersih;Labels;'
df_full_pool['review_text'] = df_full_pool['Ulasan_Bersih;Labels;'].apply(lambda x: str(x).split(';')[0].strip())
# Menghapus baris yang mungkin memiliki review_text kosong setelah ekstraksi
df_full_pool.dropna(subset=['review_text'], inplace=True)
train_reviews_set = set(df_train['review_text'].str.lower())
# Menyaring ulasan dari df_full_pool yang sudah ada di data pelatihan
df_predict_pool = df_full_pool[~df_full_pool['review_text'].str.lower().isin(train_reviews_set)].copy()

print(f"Total ulasan untuk pelatihan: {len(df_train)}")
print(f"Pool ulasan yang belum diberi label (tidak termasuk pelatihan): {len(df_predict_pool)}")

# Menyaring ulasan berbahasa Inggris dari pool prediksi
print("Mencoba memfilter ulasan berbahasa Inggris dari pool prediksi (ini mungkin memerlukan waktu)...")
df_predict_pool_english = df_predict_pool[df_predict_pool['review_text'].apply(is_english)].copy()
print(f"Jumlah ulasan berbahasa Inggris di pool prediksi: {len(df_predict_pool_english)}")

# 3. Mengambil sampel 750 data dari pool berbahasa Inggris yang sudah difilter
target_sample_size = 750
if len(df_predict_pool_english) < target_sample_size:
    print(f"Peringatan: Hanya menemukan {len(df_predict_pool_english)} ulasan Inggris yang unik untuk prediksi. Menggunakan semua yang ada.")
    sample_for_prediction = df_predict_pool_english.copy()
else:
    sample_for_prediction = df_predict_pool_english.sample(n=target_sample_size, random_state=42).reset_index(drop=True)

print(f"Berhasil mengambil sampel untuk prediksi: {len(sample_for_prediction)} ulasan.")

# Jika sample_for_prediction kosong, berhenti
if sample_for_prediction.empty:
    print("Tidak ada data untuk diprediksi.")
    exit()

# 4. Menyiapkan Data Pelatihan untuk model
X_train = df_train['review_text']
y_train = df_train['sentiment']

# 5. Ekstraksi Fitur (TF-IDF)
vectorizer = TfidfVectorizer(stop_words='english', max_features=2000)
X_train_vec = vectorizer.fit_transform(X_train)

# 6. Melatih Model (MENGGUNAKAN SVM)
svm_model = SVC(kernel='linear', random_state=42)
print("Sedang melatih model SVM...")
svm_model.fit(X_train_vec, y_train)

# 7. Memprediksi pada Data Sampel
X_test = sample_for_prediction['review_text']
X_test_vec = vectorizer.transform(X_test)
predicted_labels = svm_model.predict(X_test_vec)

# 8. Menambahkan prediksi ke dataframe
sample_for_prediction['predicted_sentiment'] = predicted_labels

# 9. Menyimpan ke CSV
output_filename = 'labeled_reviews_for_prediction_svm.csv'
sample_for_prediction.to_csv(output_filename, index=False)

# Menampilkan hasil
print(f"\nPrediksi selesai! Disimpan ke '{output_filename}'")
print("\nDistribusi Prediksi:")
print(sample_for_prediction['predicted_sentiment'].value_counts())
print("\nContoh Hasil Prediksi:")
print(sample_for_prediction[['review_text', 'predicted_sentiment']].head())


Data pelatihan dimuat. Bentuk: (250, 2)
Pool data penuh dimuat. Bentuk: (909, 1)
Total ulasan untuk pelatihan: 250
Pool ulasan yang belum diberi label (tidak termasuk pelatihan): 630
Mencoba memfilter ulasan berbahasa Inggris dari pool prediksi (ini mungkin memerlukan waktu)...
Jumlah ulasan berbahasa Inggris di pool prediksi: 12
Peringatan: Hanya menemukan 12 ulasan Inggris yang unik untuk prediksi. Menggunakan semua yang ada.
Berhasil mengambil sampel untuk prediksi: 12 ulasan.
Sedang melatih model SVM...

Prediksi selesai! Disimpan ke 'labeled_reviews_for_prediction_svm.csv'

Distribusi Prediksi:
predicted_sentiment
Positif    7
Negatif    5
Name: count, dtype: int64

Contoh Hasil Prediksi:
                                           review_text predicted_sentiment
90   engga komplit pengen cari film lama kaya the l...             Positif
97                                  boboiboy the movie             Positif
114                      incorect terus pernah benerin             Negat